
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Task 2 - Alert on Unusual Patterns
In this task, we will analyze the loan dataset for unusual patterns that may affect the performance of downstream machine learning tasks. Unusual patterns can include high cardinality in categorical features and skewed distributions in numerical features. Detecting these patterns early can help in adjusting feature engineering or model preparation steps.

**Objectives:**

- Identify columns with high cardinality, which may require transformation or encoding adjustments.
- Check for skewed distributions in numerical features, which may benefit from normalization or transformations.
- Set a flag to indicate whether unusual patterns were detected, enabling conditional paths in the MLOps pipeline.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need to use one of the following Databricks runtime(s): **17.3.x-cpu-ml-scala2.13**

## Classroom Setup

Before starting the demo, run the provided classroom setup script. This script will define configuration variables necessary for the demo. Execute the following cell:

In [0]:
%pip install mlflow

In [0]:
%run ../../Includes/Classroom-Setup-1.1demo

**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")


## Task Outline
In this task, we will:

- Load the loan dataset.
- Identify high cardinality columns.
- Detect skewed distributions in numerical columns.
- Set a flag for conditional task execution based on unusual patterns.


###Step 1: Load Loan Data
In this step, we load the loan dataset from a CSV file and inspect a sample of the data to confirm the structure.

In [0]:
# Define the dataset path
dataset_path = f"{DA.paths.datasets.banking}/banking/loan-clean.csv"

# Load the loan dataset
data = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(dataset_path)

# Display the first few rows to inspect the data
display(data)


###Step 2: Check for High Cardinality in Categorical Features
High cardinality in categorical features (e.g., features with many unique values) can complicate encoding and increase model complexity. Here, we will flag columns with high cardinality.

**Instructions:**

- Define a threshold for high cardinality (e.g., > 100 unique values).
- Identify columns that exceed this threshold.



In [0]:
# High cardinality check (threshold set to > 100 unique values)
high_cardinality_columns = [col for col in data.columns if data.select(col).distinct().count() > 100]

print("High Cardinality Columns:")
print("========================")
print(high_cardinality_columns)


###Step 3: Check for Skewed Distributions in Numerical Features
Skewed distributions in numerical features may lead to biased models and may require normalization or transformations. We will flag columns with high skewness.

**Instructions:**

- Calculate the skewness of numerical columns.
- Identify columns with an absolute skewness > 1.5 as skewed.

In [0]:
from pyspark.sql.functions import skewness, abs as abs_

skewed_columns = []
numeric_columns = [col for col, dtype in data.dtypes if dtype in ['double', 'int']]

for col in numeric_columns:
    skewness_value = data.select(skewness(col)).collect()[0][0]
    if skewness_value is not None and abs(skewness_value) > 1.5:
        skewed_columns.append(col)

print("Skewed Columns:")
print("===============")
print(skewed_columns)


###Step 4: Save Unusual Patterns Report
If unusual patterns are found, save a report that summarizes the findings. This report will help inform further data preprocessing steps.

In [0]:
# Save unusual patterns report
with open("./unusual_patterns_report.txt", "w") as f:
    f.write("Unusual Patterns Report\n")
    f.write("=======================\n")
    f.write(f"High Cardinality Columns: {high_cardinality_columns}\n")
    f.write(f"Skewed Columns: {skewed_columns}\n")

print("Unusual Patterns Report saved to ./unusual_patterns_report.txt")


###Step 5: Set Flag for Conditional Execution
If any unusual patterns are detected (high cardinality or skewed columns), we set a flag unusual_patterns_found to True. This flag will be used in the pipeline to decide whether to proceed with the regular workflow or an alternative investigation path.

In [0]:
# Import MLflow
import mlflow

# Set MLflow registry URI to Databricks Unity Catalog
mlflow.set_registry_uri('databricks-uc')

# Define a function to check for unusual patterns
def check_unusual_patterns(report_path):
    # Initialize flags
    high_cardinality_found = False
    skewed_distribution_found = False
    
    try:
        # Load the unusual patterns report
        with open(report_path, "r") as f:
            unusual_patterns_report = f.read()
        
        # Check for high cardinality columns
        if "High Cardinality Columns:" in unusual_patterns_report:
            # Check if any columns are listed under high cardinality
            high_cardinality_columns = unusual_patterns_report.split("High Cardinality Columns:")[1].split("Skewed Columns:")[0].strip()
            high_cardinality_found = bool(high_cardinality_columns and high_cardinality_columns != "None")
        
        # Check for skewed distribution columns
        if "Skewed Columns:" in unusual_patterns_report:
            # Check if any columns are listed under skewed columns
            skewed_columns = unusual_patterns_report.split("Skewed Columns:")[1].strip()
            skewed_distribution_found = bool(skewed_columns and skewed_columns != "None")
        
        # Determine unusual pattern status
        if high_cardinality_found or skewed_distribution_found:
            dbutils.jobs.taskValues.set(key="unusual_pattern_status", value="unusual_pattern_detected")
            return "unusual_pattern_detected"
        else:
            dbutils.jobs.taskValues.set(key="unusual_pattern_status", value="no_unusual_pattern_detected")
            return "no_unusual_pattern_detected"
    
    except Exception as e:
        # Handle exceptions and return a default status
        print(f"Error occurred: {e}")
        dbutils.jobs.taskValues.set(key="unusual_pattern_status", value="error_in_checking")
        return "error_in_checking"

# Example usage
dbutils.jobs.taskValues.set(key="unusual_pattern_status", value="no_unusual_pattern_detected") # Default
report_path = "./unusual_patterns_report.txt" # Path to report
pattern_status = check_unusual_patterns(report_path)
print(f"Unusual pattern status: {pattern_status}")


##Conclusion
In this notebook, we:

- Analyzed the loan dataset for unusual patterns, including high cardinality and skewed distributions.
- Saved a report with the identified patterns for reference.
- Set a flag to enable conditional execution in the pipeline based on the detection of unusual patterns.

This task ensures that any detected unusual patterns are accounted for in the MLOps workflow, allowing for tailored data processing and troubleshooting steps. This flag will guide the workflow to either save the final report or proceed with an investigation of the unusual patterns.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>